# Results

In [75]:
PROBLEMS_FILE = './data/problems.json'
SOLUTIONS_FILE = './data/solutions.json'
EVALUATIONS_FILE = './data/evaluations.json'

In [76]:
from src import read_json

problems = read_json(PROBLEMS_FILE)
solutions = read_json(SOLUTIONS_FILE)
evaluations = read_json(EVALUATIONS_FILE)

## FP2MP-Eval

In [77]:
import pandas as pd

data = []

for evaluation_id, evaluation_data in evaluations.items():
    solution_id = evaluation_data['solution_id']
    solution_baseline = solutions[solution_id]['baseline']
    problem_id = solutions[solution_id]['problem_id']
    for evaluation in evaluation_data['content']:
        for criterion in evaluation.keys():
            score = evaluation[criterion]['score']
            data.append({
                'baseline': solution_baseline,
                'problem_id': problem_id,
                'criterion': criterion,
                'score': score
            })

df = pd.DataFrame(data)
df

,baseline,problem_id,criterion,score
0,single-agent,82fbc85fcfff5fe780b66e3914768cd3,framing,4
1,single-agent,82fbc85fcfff5fe780b66e3914768cd3,decomposition,4
2,single-agent,82fbc85fcfff5fe780b66e3914768cd3,diversity,4
3,single-agent,82fbc85fcfff5fe780b66e3914768cd3,coherence,4
4,single-agent,82fbc85fcfff5fe780b66e3914768cd3,justification,4
...,...,...,...,...
11995,major-vote,72fd7da3ab53563a88b22ec77cad6aad,coherence,3
11996,major-vote,72fd7da3ab53563a88b22ec77cad6aad,justification,2
11997,major-vote,72fd7da3ab53563a88b22ec77cad6aad,uncertainty_handling,1
11998,major-vote,72fd7da3ab53563a88b22ec77cad6aad,knowledge_integration,3


In [78]:
df.groupby(['baseline', 'criterion'])['score'].agg(lambda s : f'{s.mean().round(2)} ± {s.std().round(2)}').unstack(level=0)

baseline,chain-of-thoughts,debate,generator-critic,major-vote,single-agent
criterion,,,,,
coherence,4.13 ± 0.34,4.17 ± 0.47,4.44 ± 0.5,3.94 ± 0.61,4.17 ± 0.38
decomposition,4.09 ± 0.29,4.13 ± 0.43,4.38 ± 0.49,3.89 ± 0.62,4.14 ± 0.35
diversity,2.81 ± 0.64,3.1 ± 0.62,3.41 ± 0.56,3.04 ± 0.8,3.32 ± 0.62
framing,4.05 ± 0.28,4.15 ± 0.45,4.44 ± 0.5,3.92 ± 0.64,4.19 ± 0.39
justification,3.16 ± 0.36,3.37 ± 0.55,3.91 ± 0.35,3.09 ± 0.64,3.45 ± 0.5
knowledge_integration,4.13 ± 0.4,4.22 ± 0.5,4.53 ± 0.5,3.97 ± 0.67,4.28 ± 0.45
metacognition,2.51 ± 0.59,2.5 ± 0.7,3.51 ± 0.53,2.26 ± 0.87,2.6 ± 0.62
uncertainty_handling,3.04 ± 0.54,2.75 ± 0.75,3.83 ± 0.47,2.46 ± 0.86,2.8 ± 0.6


In [79]:
df.groupby(['baseline']).agg({'score':lambda s : f'{s.mean().round(2)} ± {s.std().round(2)}'}).transpose()

baseline,chain-of-thoughts,debate,generator-critic,major-vote,single-agent
score,3.49 ± 0.78,3.55 ± 0.87,4.06 ± 0.64,3.32 ± 0.98,3.62 ± 0.8


## Token consumption

In [80]:
data = []

for solution_id, solution_data in solutions.items():
    solution_baseline = solution_data['baseline']
    solution_log = solution_data['log']

    completion_tokens = 0
    prompt_tokens = 0
    total_tokens = 0

    for message in solution_log:
        if 'response_metadata' in message:
            response_metadata = message['response_metadata']
            if 'token_usage' in response_metadata:
                token_usage = response_metadata['token_usage']

                completion_tokens += token_usage.get('completion_tokens', 0)
                prompt_tokens += token_usage.get('prompt_tokens', 0)
                total_tokens += token_usage.get('total_tokens', 0)

    data.append({
        'baseline': solution_baseline,
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
        'total_tokens': total_tokens
    })

df = pd.DataFrame(data)
df

,baseline,prompt_tokens,completion_tokens,total_tokens
0,single-agent,24,2070,2094
1,chain-of-thoughts,1535,2918,4453
2,generator-critic,4554,7406,11960
3,debate,50438,27667,78105
4,major-vote,5539,9910,15449
...,...,...,...,...
295,single-agent,21,2006,2027
296,chain-of-thoughts,1269,2499,3768
297,generator-critic,5291,7948,13239
298,debate,56789,26546,83335


In [82]:
df.groupby('baseline').agg(lambda s : f'{s.mean().astype(int)} ± {s.std().astype(int)}')

,prompt_tokens,completion_tokens,total_tokens
baseline,,,
chain-of-thoughts,1444 ± 230,2904 ± 619,4349 ± 779
debate,54285 ± 6063,27717 ± 3198,82003 ± 9124
generator-critic,4572 ± 649,6865 ± 879,11438 ± 1455
major-vote,5201 ± 542,8527 ± 1325,13728 ± 1682
single-agent,20 ± 2,2130 ± 313,2151 ± 313
